In [1]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import csv
from tqdm import tqdm
from PIL import Image
from utils import preprocess_pil

In [2]:
# to deal with metadata
raw_root = "../data_raw/Derm7pt"
out_root = "../data_ready/Derm7pt"
meta_raw = pd.read_csv(f"{raw_root}/metadata/meta.csv")
meta = meta_raw.copy()
meta = meta.drop(['case_num', 'seven_point_score','level_of_diagnostic_difficulty', 
                  'elevation', 'location', 'sex', 'management', 'clinic', 'case_id', 'notes'], axis=1)
print(meta.columns)

map_dict = {}
map_dict['diagnosis'] = {"clark nevus": "Non-Melanoma", "reed or spitz nevus": "Non-Melanoma", "seborrheic keratosis": "Non-Melanoma",
                      "basal cell carcinoma": "Non-Melanoma", "dermal nevus": "Non-Melanoma", "vascular lesion": "Non-Melanoma", 
                      "blue nevus": "Non-Melanoma", "lentigo": "Non-Melanoma", "dermatofibroma": "Non-Melanoma",
                      "congenital nevus": "Non-Melanoma", "melanosis": "Non-Melanoma", "combined nevus": "Non-Melanoma",
                       "miscellaneous": "Non-Melanoma", "recurrent nevus": "Non-Melanoma", 
                       "melanoma (less than 0.76 mm)": "Melanoma", "melanoma (in situ)": "Melanoma", "melanoma (0.76 to 1.5 mm)": "Melanoma", 
                       "melanoma (more than 1.5 mm)": "Melanoma", "melanoma metastasis": "Melanoma", "melanoma": "Melanoma"}

map_dict['pigmentation'] = {"diffuse irregular": "irregular", "diffuse regular": "regular", 
                            "localized irregular": "irregular", "localized regular": "regular"}

map_dict['regression_structures'] = {"blue areas": "present", "combinations": "present", "white areas": "present"}

map_dict['vascular_structures'] = {"dotted": "atypical", "linear irregular": "atypical", "within regression": "typical", 
                                   "arborizing": "typical", "comma": "typical", "hairpin": "typical", "wreath": "typical"}

for column in ['diagnosis', 'pigmentation', 'regression_structures', 'vascular_structures']:
    meta[column] = meta[column].replace(map_dict[column])

for column in meta.columns[:-2]:
    print(meta[column].value_counts())

# to extract class anotation, class names, image paths and save them into the raw metadata
image_classes = np.array(meta["diagnosis"])

class_names = np.array(['Non-Melanoma', 'Melanoma'], dtype=object)
# class_names = np.unique(image_classes)
class_names_2save = np.column_stack([np.arange(1, len(class_names) + 1), class_names])
np.savetxt(f"{raw_root}/metadata/classes.txt", class_names_2save, fmt=['%d', '%s'], delimiter=' ', encoding='utf-8')

image_classes_2save = np.column_stack([np.arange(1, len(image_classes) + 1), image_classes])
np.savetxt(f"{raw_root}/metadata/image_classes.txt", image_classes_2save, fmt=['%d', '%s'], delimiter=' ', encoding='utf-8')

image_paths = np.array(meta["derm"])
image_paths_2save = np.column_stack([np.arange(1, len(image_paths) + 1), image_paths])
np.savetxt(f"{raw_root}/metadata/image_paths.txt", image_paths_2save, fmt=['%d', '%s'], delimiter=' ', encoding='utf-8')

meta = meta.drop(["diagnosis", "derm"], axis=1)

# get one-hot instance-level concepts annotations
image_concepts = pd.get_dummies(meta, dtype=np.int32)
np.savetxt(f"{raw_root}/metadata/image_concepts.txt", image_concepts.values, fmt='%d', delimiter=' ')

# save concepts names
concept_names = np.array(image_concepts.columns)
concept_names_2save = np.column_stack([np.arange(1, len(concept_names) + 1), concept_names])
np.savetxt(f"{raw_root}/metadata/concepts.txt", concept_names_2save, fmt=['%d', '%s'], delimiter=' ', encoding='utf-8')

Index(['diagnosis', 'pigment_network', 'streaks', 'pigmentation',
       'regression_structures', 'dots_and_globules', 'blue_whitish_veil',
       'vascular_structures', 'derm'],
      dtype='object')
diagnosis
Non-Melanoma    759
Melanoma        252
Name: count, dtype: int64
pigment_network
absent      400
typical     381
atypical    230
Name: count, dtype: int64
streaks
absent       653
irregular    251
regular      107
Name: count, dtype: int64
pigmentation
absent       588
irregular    305
regular      118
Name: count, dtype: int64
regression_structures
absent     758
present    253
Name: count, dtype: int64
dots_and_globules
irregular    448
regular      334
absent       229
Name: count, dtype: int64
blue_whitish_veil
absent     816
present    195
Name: count, dtype: int64


In [3]:
jpeg_quality = 95
image_size = 224
shortest_side = 256

raw_root = Path(raw_root)
out_root = Path(out_root)
images_raw = Path(raw_root)/"images"
images_out = out_root / "images"
meta_out = out_root / "metadata"
annot_out = out_root / "annotations"
images_out.mkdir(parents=True, exist_ok=True)
meta_out.mkdir(parents=True, exist_ok=True)
annot_out.mkdir(parents=True, exist_ok=True)

# put instance from samw class together
order_key = np.full(image_classes.shape, 2, dtype=int)
order_key[image_classes == 'Non-Melanoma'] = 0
order_key[image_classes == 'Melanoma'] = 1
perm = np.argsort(order_key, kind='stable')
image_classes  = image_classes[perm]
image_paths = image_paths[perm]
image_concepts = image_concepts.values[perm, :]

sample_rows = []
labels_onehot = []
counter = 0
C = len(class_names)
D = len(concept_names)
class_name2id = {"Non-Melanoma": 0, "Melanoma": 1}

for cls_name, img_path in tqdm(zip(image_classes, image_paths), desc="Preprocessing images"):
    class_id = class_name2id[cls_name]
    try:
        with Image.open(images_raw / img_path) as img:
            img = preprocess_pil(img, target_size=image_size, shortest=shortest_side)
    except Exception as e:
        print(f"[WARN] Skipping corrupted/invalid image: {img_path} ({e})")
        continue
    counter += 1
    cls_code = f"{class_id:01d}"
    global_code = f"{counter:05d}"
    sid = f"{cls_code}_{global_code}"

    out_file = images_out / f"{sid}.jpg"
    # if out_file.exists():
    #     # If re-running, skip existing files
    #     continue
    img.save(out_file, format="JPEG", quality=jpeg_quality, optimize=False)

    rel_path = f"images/{sid}.jpg"
    sample_rows.append((sid, rel_path, class_id, cls_name))

    y = np.zeros((2,), dtype=np.float32)
    y[class_id] = 1.0
    labels_onehot.append(y)

if not sample_rows:
    raise RuntimeError("All images failed during preprocessing. Check raw data integrity.")

labels_onehot = np.stack(labels_onehot, axis=0)           # (N, C)
concepts_per_image =  image_concepts # (N, D)



# Write metadata CSVs
samples_csv = meta_out / "samples.csv"
with samples_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["sample_id", "rel_path", "class_id", "class_name"])
    w.writerows(sample_rows)

classes_csv = meta_out / "classes.csv"
with classes_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["class_id", "class_name"])
    for i, name in enumerate(class_names):
        w.writerow([i, name])

concepts_csv = meta_out / "concepts.csv"
with concepts_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["concept_id", "concept_name"])
    for i, name in enumerate(concept_names):
        w.writerow([i, name])

# clc_csv = meta_out / "class_level_concepts.csv"
# with clc_csv.open("w", newline="", encoding="utf-8") as f:
#     w = csv.writer(f)
#     header = ["class_id"] + [f"c{i+1}" for i in range(D)]
#     w.writerow(header)
#     for i in range(C):
#         w.writerow([i] + list(map(int, class_level_concepts[i])))

ilc_csv = meta_out / "instance_level_concepts.csv"
with ilc_csv.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    header = ["instance_id"] + [f"c{i+1}" for i in range(D)]
    w.writerow(header)
    for i in range(len(concepts_per_image)):
        w.writerow([i] + list(map(int, concepts_per_image[i])))


stats = {
    "num_images": int(labels_onehot.shape[0]),
    "num_classes": int(C),
    "num_concepts": int(D),
    "class_hist": {
        f"{i}-{class_names[i]}": int(sum(1 for row in sample_rows if row[2] == i)) for i in range(C)
    },
    "image_size": image_size,
    "shortest_side": shortest_side,
}
with (meta_out / "stats.json").open("w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print(f"[OK] Wrote {labels_onehot.shape[0]} images.")
print(f" - images:   {images_out}")
print(f" - metadata: {meta_out}")

np.save(annot_out / "labels_onehot.npy", labels_onehot)            # (N, C)
np.save(annot_out / "concepts_per_image.npy", concepts_per_image)  # (N, D)
print(f" - annotations: {annot_out} (labels_onehot.npy, concepts_per_image.npy)")
print(labels_onehot.shape, concepts_per_image.shape)

Preprocessing images: 1011it [00:05, 170.08it/s]

[OK] Wrote 1011 images.
 - images:   ..\data_ready\Derm7pt\images
 - metadata: ..\data_ready\Derm7pt\metadata
 - annotations: ..\data_ready\Derm7pt\annotations (labels_onehot.npy, concepts_per_image.npy)
(1011, 2) (1011, 19)
